# Highest-Power Coefficient in the EMR Curvature Formula

This notebook reproduces the finite-dimensional computation behind
Theorem 3.4 of *Diffeomorphism Groups of Circle Bundles over Integral
Symplectic Manifolds*.  The program alternates over all permutations
and contracts copies of the standard complex structure.

The complete research code is available in
[EMR-Paper-Computation](https://github.com/egisatoshi/EMR-Paper-Computation).


## Standard complex structure

On a real vector space of dimension $2k$, let

$$
J=\begin{pmatrix}0&I_k\\-I_k&0\end{pmatrix}.
$$

The notebook uses $k=2$ for a quick interactive calculation.  The
final section records the larger computations with exactly the same
definitions.


In [1]:
def k := 2

def J :=
  generateTensor
    (\match as list integer with
       | [$i, #(i + k)] -> 1
       | [$i, #(i - k)] -> -1
       | _ -> 0)
    [2 * k, 2 * k]


In [2]:
J


$\begin{pmatrix} 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \\ -1 & 0 & 0 & 0 \\ 0 & -1 & 0 & 0 \\ \end{pmatrix}$

In [3]:
withSymbols [a, b, c] J_a~c . J_c~b


$\begin{pmatrix} -1 & 0 & 0 & 0 \\ 0 & -1 & 0 & 0 \\ 0 & 0 & -1 & 0 \\ 0 & 0 & 0 & -1 \\ \end{pmatrix}$

## Alternating product

Let $S_k$ be the signed sum of products

$$
S_k=\sum_{\sigma\in S_{2k}}\operatorname{sgn}(\sigma)
    \prod_{i=1}^{k}J_{\sigma(2i-1),\sigma(2i)}.
$$

`evenAndOddPermutations` supplies the two signs separately.


In [4]:
def S :=
  let (es, os) := evenAndOddPermutations (2 * k) in
    sum (map (\σ -> product (map (\i -> J_(σ (2 * i - 1))_(σ (2 * i))) (between 1 k))) es) -
    sum (map (\σ -> product (map (\i -> J_(σ (2 * i - 1))_(σ (2 * i))) (between 1 k))) os)


In [5]:
S


$-8$

## Curvature coefficient

The highest power of the bundle parameter $p$ is built from

$$
T_{abc}{}^d=-J_{bc}J_a{}^d+J_{ac}J_b{}^d
             +2J_{ab}J_c{}^d.
$$

The indices $a_0,\ldots,a_{k-1}$ form a cyclic chain.  Tensor product
followed by `.` contracts that chain, while the outer fold adds the
even and odd permutation contributions.


In [6]:
def T_a_b_c~d :=
  -1 * J_b_c . J_a~d +
  J_a_c . J_b~d +
  2 * J_a_b . J_c~d

def S' :=
  withSymbols [a]
    let (es, os) := evenAndOddPermutations (2 * k) in
      (\xs -> foldl (+) (head xs) (tail xs))
        (map
          (\σ ->
            (\xs -> foldl (.) (head xs) (tail xs))
              (map (\i -> T_(σ (2 * i - 1))_(σ (2 * i))_(a_(modulo i k))~(a_(i - 1))) (between 1 k)))
          es) -
      (\xs -> foldl (+) (head xs) (tail xs))
        (map
          (\σ ->
            (\xs -> foldl (.) (head xs) (tail xs))
              (map (\i -> T_(σ (2 * i - 1))_(σ (2 * i))_(a_(modulo i k))~(a_(i - 1))) (between 1 k)))
          os)


In [7]:
S'


$192$

## Higher dimensions

The same program gives

$$
\begin{array}{c|rr}
k&S_k&S'_k\\ \hline
2&-8&192\\
3&-48&0\\
4&384&61440
\end{array}
$$

On Egison 5.1.0 the full $k=4$ run takes about 25 minutes.  Keeping
the interactive notebook at $k=2$ makes every cell quick to rerun;
changing the first definition to `def k := 4` performs the complete
Theorem 3.4 computation without any other code changes.
